# gnomAD population-frequency EDA

gnomAD v4 (`gnomad_r4`) allele frequencies at expert-reviewed ClinVar SNVs.

Pipeline:

```
python src/ingest_gnomad.py       # data/raw/gnomad_clinvar_sites.parquet
python src/clean_gnomad.py        # data/processed/gnomad_clean.parquet
python src/join_clinvar_gnomad.py # + --vus
```

Preferred AF is **joint** (exomes+genomes), then exome, then genome. Sites absent from gnomAD stay `in_gnomad=False`; modeling fills those AFs to 0 (ACMG PM2-style).

**Circularity:** ClinVar pathogenic/benign labels already use population frequency (BA1/BS1/PM2). A large AF-vs-label split is expected and is not independent biological signal.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

In [ ]:
load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])
sys.path.insert(0, str(project_root / "src"))

from features import add_gnomad_features, prepare_modeling_frame

GNOMAD_CLEAN = project_root / "data/processed/gnomad_clean.parquet"
JOINED = project_root / "data/processed/clinvar_uniprot_gnomad_position_matched.parquet"

In [ ]:
gnomad = pd.read_parquet(GNOMAD_CLEAN)
gnomad.shape, gnomad["in_gnomad"].mean()

In [ ]:
print(gnomad["in_gnomad"].value_counts().to_string())
print("filter_pass:", gnomad["gnomad_filter_pass"].value_counts().to_dict())
found = gnomad.loc[gnomad["in_gnomad"], "gnomad_af"]
found.describe(percentiles=[0.5, 0.9, 0.99])

## AF vs ClinVar label

One row per labelled variant after the UniProt position-match + gnomAD join.

In [ ]:
joined = pd.read_parquet(JOINED)
model = prepare_modeling_frame(joined)
print(model.shape)
model.groupby("label").agg(
    n=("VariationID", "size"),
    pct_in_gnomad=("in_gnomad", "mean"),
    median_af=("gnomad_af", "median"),
    mean_log10_af=("log10_gnomad_af", "mean"),
    pct_filter_pass=("gnomad_filter_pass", "mean"),
)

In [ ]:
print(model["gnomad_af_bin"].value_counts().to_string())
pd.crosstab(model["gnomad_af_bin"], model["label"], normalize="index").round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.kdeplot(
    data=model, x="log10_gnomad_af", hue="label",
    common_norm=False, ax=ax,
)
ax.set_xlabel("log10(gnomAD AF + 1e-6)")
ax.set_title("Population frequency by ClinVar label")
plt.tight_layout()
plt.show()

## Feature engineering (modeling matrix)

Added to `src/features.py`:

| column | type | notes |
|---|---|---|
| `log10_gnomad_af` | numeric | joint AF, absent → 0 before log |
| `log10_gnomad_af_popmax` | numeric | filtering AF (FAF95) when available |
| `log1p_gnomad_nhomalt` | numeric | log1p(homozygote count) |
| `in_gnomad` | bool | present in gnomAD v4 sites |
| `gnomad_filter_pass` | bool | empty FILTER on available sources |
| `gnomad_af_bin` | categorical | absent / singleton_or_private / ultra_rare / rare / low_freq / common_bs1 / common_ba1 |

`gnomad_an` is kept on the joined table for EDA but **not** used as a model feature (gene-coverage proxy).